In [1]:
import os
import re
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import warnings
import xarray as xr
from scipy import sparse


/jp_envs/envs/scanpy/lib/python3.12/site-packages/leidenalg/VertexPartition.py:413: SyntaxWarning: invalid escape sequence '\m'
  .. math:: Q = \\frac{1}{m} \\sum_{ij} \\left(A_{ij} - \\frac{k_i^\mathrm{out} k_j^\mathrm{in}}{m} \\right)\\delta(\\sigma_i, \\sigma_j),
/jp_envs/envs/scanpy/lib/python3.12/site-packages/leidenalg/VertexPartition.py:788: SyntaxWarning: invalid escape sequence '\m'
  .. math:: Q = \\sum_{ij} \\left(A_{ij} - \\gamma \\frac{k_i^\mathrm{out} k_j^\mathrm{in}}{m} \\right)\\delta(\\sigma_i, \\sigma_j),
/jp_envs/envs/scanpy/lib/python3.12/site-packages/leidenalg/Optimiser.py:27: SyntaxWarning: invalid escape sequence '\g'
  implementation therefore does not guarantee subpartition :math:`\gamma`-density.
/jp_envs/envs/scanpy/lib/python3.12/site-packages/leidenalg/Optimiser.py:346: SyntaxWarning: invalid escape sequence '\s'
  .. math:: Q = \sum_k \\lambda_k Q_k.


In [ ]:
## load RNA matrix and metadata
combined=sc.read_h5ad("tmp_dataset/combined_data_rna.raw.h5ad") ## load the raw h5ad data with all cells raw RNA values
sc.pp.normalize_total(combined, target_sum=1e4)
sc.pp.log1p(combined)
celltype_meta = pd.read_csv(
    "../input_metadata/cell_meta.tsv",
    sep="\t",
    index_col=0
)

combined.obs["cell_type"] = combined.obs.index.map(celltype_meta["cell_type"])

for celltype in ["epi", "t", "fib","mye","end","smc","b"]:
    combined.obs[f"{celltype}_cluster"] = (
        combined.obs.index.map(cluster_meta[f"{celltype}_cl"])
    )


In [ ]:
### calculate DEG for cell types
combined1 = combined[combined.obs["cell_type"].notna(), :].copy()
anno_col = "cell_type"

sc.tl.rank_genes_groups(combined1, method = 'wilcoxon', groupby = anno_col, pts = True)

DEG = sc.get.rank_genes_groups_df(combined1, group = None)
DEG = DEG.rename(columns={'group': 'cell_type'})

DEG.to_csv(f'tmp_dataset/DEG/DEG_cell_type.csv', index = False)
DEG.head(n=10)

In [ ]:
### calculate DEG for G1/G2/G3 comparison
combined1 = combined
ct = "mye"  ### taking Myeloid as example
cell_meta = pd.read_csv("../input_metadata/cell_meta.tsv",sep="\t")

subcls = cell_meta[f"{ct}_cl"].dropna().unique()

for cl in subcls:
    print(f"Processing cell type: {cl}")
    comparisons = [
        (f"G2_{cl}", f"G1_{cl}"),
        (f"G3_{cl}", f"G1_{cl}"),
        (f"G3_{cl}", f"G2_{cl}")
    ]
    output_file = f"tmp_dataset/DEG/DEG_subset_{ct}_{cl}_group_comparison.csv"
    
    meta_tmp = cell_meta.copy()
    
    meta_tmp['cell_group'] = np.where(
        meta_tmp[f'{ct}_cl'] == cl,
        "G" + meta_tmp['Group'].astype(str) + f"_{cl}",
        np.nan
    )
    
    meta_tmp = meta_tmp.dropna(subset=['cell_group'])
    meta_tmp = meta_tmp.set_index('cell_id')

    combined_tmp = combined.copy()
    common_cells = combined_tmp.obs.index.intersection(meta_tmp.index)
    combined_tmp = combined_tmp[common_cells].copy()
    combined_tmp.obs['cell_group'] = meta_tmp.loc[common_cells, 'cell_group']
    
    print(combined_tmp.obs.shape)  # sanity check
    
    combined_tmp = combined_tmp[combined_tmp.obs["cell_group"].notna(), :].copy()
    combined_tmp.obs["cell_group"] = combined_tmp.obs["cell_group"].astype("category")
    
    anno_col = "cell_group"
    
    deg_list = []

    for target_group, reference_group in comparisons:
        print(f"  DEG: {target_group} vs {reference_group}")

        groups_present = combined_tmp.obs[anno_col].unique()
        if target_group not in groups_present or reference_group not in groups_present:
            print(f"  Skip {target_group} vs {reference_group} (missing group)")
            continue

        adata_sub = combined_tmp[
            combined_tmp.obs['cell_group'].isin([target_group, reference_group])
        ].copy()

        adata_sub.obs['cell_group'] = adata_sub.obs['cell_group'].astype('category')
        adata_sub.obs['cell_group'] = adata_sub.obs['cell_group'].cat.reorder_categories(
            [target_group, reference_group]
        )

        sc.tl.rank_genes_groups(
            adata_sub,
            groupby='cell_group',
            method='wilcoxon',
            pts=True
        )

        DEG = sc.get.rank_genes_groups_df(adata_sub, group=target_group)
        DEG = DEG.rename(columns={'group': 'cell_group'})

        DEG['target'] = target_group
        DEG['reference'] = reference_group
        DEG['comparison'] = f"{target_group}_vs_{reference_group}"

        deg_list.append(DEG)

    if len(deg_list) == 0:
        print(f"No valid comparisons for {cl}")
        continue

    pairwise_DEG = pd.concat(deg_list, ignore_index=True)
    pairwise_DEG.to_csv(output_file, index=False)
    
    print(f"DEG analysis complete for {cl}. Total {len(pairwise_DEG)} genes identified.")